# ⚡ Benchmark 5 Model Prediksi Beban Listrik Panama
**Chronos-2 · TimesFM 2.5 · Moirai 2.0 · Lag-Llama · SARIMAX** (+ Seasonal Naive sebagai patokan)

Notebook ini menjalankan semua model dengan **desain evaluasi yang sama**, lalu menyimpan hasilnya ke folder `results/` yang dibaca dashboard Streamlit.

| Horizon | Jendela uji | Periode |
|---|---|---|
| 24 jam | 7 jendela | 7 hari terakhir |
| 168 jam | 4 jendela | 4 minggu terakhir |
| 720 jam | 1 jendela | 30 hari terakhir |

**Sebelum mulai:** `Runtime → Change runtime type → T4 GPU`. Total waktu ± 20–30 menit (sebagian besar untuk instalasi).

Setiap model dijalankan di *virtual environment* terpisah karena versi library-nya saling bentrok (misalnya Moirai butuh `torch 2.4` & `numpy 1.26`, sedangkan TimesFM butuh versi lebih baru). Jadi Anda **tidak perlu restart runtime** di antara model.

## 1. Ambil kode & data dari GitHub

In [ ]:
REPO_URL = "https://github.com/inaswulanr/beban-listrik-panama.git"   # ganti jika nama repo berbeda

import os
if not os.path.exists("/content/repo"):
    !git clone -q $REPO_URL /content/repo
%cd /content/repo
!ls data modeling
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Tidak ada GPU - model tetap jalan di CPU tetapi lebih lambat"

In [ ]:
# Helper: buat virtualenv terpisah per model dengan uv (cepat & berbagi cache unduhan)
!pip install -q -U uv
import os, subprocess

def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print("❌ PERINTAH GAGAL:", " ".join(cmd))
        print(r.stdout[-3000:])
        print(r.stderr[-3000:])
        raise RuntimeError(f"Instalasi gagal (kode {r.returncode}). Salin pesan di atas untuk diagnosis.")
    return r

def make_env(name, python="3.12"):
    req = f"/content/repo/modeling/requirements/{name}.txt"
    if not os.path.exists(req):
        raise FileNotFoundError(f"{req} tidak ditemukan. Pastikan folder modeling/requirements/ ikut terunggah ke GitHub.")
    env = f"/content/envs/{name}"
    if not os.path.exists(f"{env}/.siap"):          # penanda: instalasi sebelumnya sukses
        print(f"Menginstal library untuk {name} ... (1-5 menit)")
        _run(["uv", "venv", "--clear", "--python", python, env])
        _run(["uv", "pip", "install", "--python", f"{env}/bin/python", "-r", req])
        open(f"{env}/.siap", "w").close()
        print(f"✅ {name} siap")
    return f"{env}/bin/python"

## 2. SARIMAX + Seasonal Naive (statistik klasik)
SARIMAX(2,0,1)(1,1,1,24) dengan variabel eksogen kalender (Fourier mingguan, hari libur, masa sekolah). ± 3–5 menit di CPU.

In [ ]:
!python modeling/run_sarimax.py
# Opsional: tambahkan suhu aktual sebagai variabel X (cuaca 'oracle', tidak adil vs model lain)
# !python modeling/run_sarimax.py --weather

## 3. Chronos-2 (Amazon)

In [ ]:
py = make_env("chronos")
!{py} modeling/run_chronos.py
# Versi lama (Chronos-Bolt): !{py} modeling/run_chronos.py --model_id amazon/chronos-bolt-base

## 4. TimesFM 2.5 (Google)

In [ ]:
py = make_env("timesfm")
!{py} modeling/run_timesfm.py

## 5. Moirai 2.0 (Salesforce)

In [ ]:
py = make_env("moirai")
!{py} modeling/run_moirai.py

## 6. Lag-Llama

In [ ]:
if not os.path.exists("lag-llama"):
    !git clone -q https://github.com/time-series-foundation-models/lag-llama.git
py = make_env("lagllama")
!{py} modeling/run_lagllama.py --repo_dir lag-llama

## 7. Bandingkan semua model (MAPE & RMSE)

In [ ]:
!python modeling/evaluate.py

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
m = pd.read_csv("results/metrics.csv")
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
for a, col in zip(ax, ["MAPE (%)", "RMSE"]):
    piv = m.pivot(index="model", columns="horizon_h", values=col)
    piv.sort_values(168).plot.barh(ax=a)
    a.set_title(f"{col} - lebih kecil lebih baik"); a.set_xlabel(col); a.legend(title="Horizon (jam)")
plt.tight_layout(); plt.show()
m.pivot(index="model", columns="horizon", values="MAPE (%)").round(2)

## 8. Unggah hasil ke GitHub
Unduh `results.zip`, ekstrak, lalu di GitHub buka folder **`results/`** → **Add file → Upload files** → seret `metrics.csv`, `forecasts.csv`, dan isi folder `raw/` → **Commit**.
Streamlit Cloud akan otomatis memuat ulang dashboard dengan hasil terbaru.

In [ ]:
!cd /content/repo && zip -qr /content/results.zip results
from google.colab import files
files.download("/content/results.zip")